### Proyecto delitos (Cursor en Python)
#### Conexión a la base de datos con SQL Server

In [1]:
import pyodbc # Importación de librería pyodbc para realizar la conexión

In [ ]:
# Creación de la conexión
conn_string= ('DRIVER={ODBC Driver 17 for SQL Server};' # Manejador de base de datos en SQL Server
              'SERVER= USER_SERVER;'                    # Nombre del servidor
              'DATABASE= USER_DATABASE;'                # Nombre de la base de datos
              'UID= USER_UID;'                          # Usuario y contraseña
              'PWD= USER_PWD')         
conn= pyodbc.connect(conn_string)              # Se establece la conexión
cursor= conn.cursor()                          # Se crea un objeto cursor

In [ ]:
# Consulta para contar los registros
cursor.execute('SELECT count(*) from delitos') # Instrucción que ejecuta una consulta en SQL Server
total= cursor.fetchone()[0]                    # Recupera el primer elemento de la fila de la consulta 
print('Total de registros: ', total)

Total de registros:  2562994


In [ ]:
# Imprime una muestra de la base de datos
cursor.execute('SELECT top 5 * from delitos')
rows= cursor.fetchall()                        # Recupera todas las filas de la consulta
for row in rows:
    print(row)

(2015, 5, 'Coahuila de Zaragoza', 5024, 'Parras', 'La vida y la Integridad corporal', 'Feminicidio', 'Feminicidio', 'Con arma de fuego', 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1)
(2015, 1, 'Aguascalientes', 1001, 'Aguascalientes', 'La vida y la Integridad corporal', 'Homicidio', 'Homicidio doloso', 'Con arma de fuego', 2, 0, 1, 1, 0, 1, 1, 0, 2, 1, 0, 1, 2)
(2015, 7, 'Chiapas', 7032, 'Escuintla', 'El patrimonio', 'Robo', 'Robo de maquinaria', ' Robo de cables tubos y otros objetos destinados a servicios públicos Sin violencia', 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3)
(2015, 13, 'Hidalgo', 13060, 'Tenango de Doria', 'La vida y la Integridad corporal', 'Lesiones', 'Lesiones culposas', 'Con otro elemento', 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 4)
(2015, 12, 'Guerrero', 12028, 'Chilapa de Álvarez', 'La familia', 'Otros delitos contra la familia', 'Otros delitos contra la familia', 'Otros delitos contra la familia', 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5)


#### Limpieza y transformación de datos

In [ ]:
# Conteo de valores nulos
cursor.execute('SELECT count(*) from delitos where Anio is null or Entidad is null or Municipio is null or Bien_juridico_afectado is null or ' \
                'Tipo_de_delito is null or Subtipo_de_delito is null')
total=cursor.fetchone()[0]     
print('Total: ', total)

Total:  0


In [ ]:
# Conteo de filas duplicadas por medio de concatenación de columnas
cursor.execute('SELECT concat(Anio, Entidad, Municipio, Bien_juridico_afectado, Tipo_de_delito, Subtipo_de_delito, Modalidad), ' \
                'count(*) as Conteo from delitos ' \
                'group by concat(Anio, Entidad, Municipio, Bien_juridico_afectado, Tipo_de_delito, Subtipo_de_delito, Modalidad)' \
                'having count(*) > 1')
rows= cursor.fetchall()
for row in rows:
    print(row)

In [ ]:
# Creación de una nueva tabla normalizada con el conteo de delitos por mes
# Se agrega la clave foránea id_delito que se relaciona con el id creado en SQL Server 

cursor.execute('CREATE table meses (id_delito int not null, Mes varchar(12), Conteo int not null,' \
                'FOREIGN KEY(id_delito) REFERENCES delitos(Id) )')
conn.commit()                                          # Ejecuta los cambios hechos (la definición de la tabla)

In [ ]:

lectura= ('DRIVER={ODBC Driver 17 for SQL Server};' 
              'SERVER=ACERTERE;'                    
              'DATABASE=delitos_municipal;'           
              'UID=analista1;'                          
              'PWD=123')

escritura= ('DRIVER={ODBC Driver 17 for SQL Server};'
              'SERVER=ACERTERE;'     
              'DATABASE=delitos_municipal;'      
              'UID=analista1;'                          
              'PWD=123')         
        
conn_lectura= pyodbc.connect(lectura)                   # Creación de dos conexiones para lectura y escritura
conn_escritura= pyodbc.connect(escritura) 

In [ ]:
cursor_lectura= conn_lectura.cursor()                   # Creación de los cursores para lectura y escritura
cursor_escritura= conn_escritura.cursor()

cursor_escritura.fast_executemany= True                 # Mejora el rendimiento de executemany (Inserción de registros)

cursor_lectura.execute('SELECT Id, Enero, Febrero, Marzo, Abril, Mayo, Junio, Julio, Agosto, Septiembre, Octubre, Noviembre, Diciembre from delitos')
while True:                                             # Ciclo para leer y almacenar 10,000 registros en bloques
    rows= cursor_lectura.fetchmany(10000)
    if not rows:                                        
        break
    data= []                                            # Arreglo donde se almacenan las variables id_delito, meses y valores
    for row in rows:
        id_delito= row.Id
        meses= ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio',                 # Meses en columnas de la tabla delitos
                'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']
        valores= [row.Enero, row.Febrero, row.Marzo, row.Abril, row.Mayo, row.Junio, row.Julio, # Almacena los valores de cada columna en filas
                    row.Agosto, row.Septiembre, row.Octubre, row.Noviembre, row.Diciembre] 
        for mes, conteo in zip(meses, valores):
            data.append((id_delito, mes, conteo))
    cursor_escritura.executemany('INSERT INTO meses (id_delito, Mes, Conteo) values (?,?,?)', data)     
    conn_escritura.commit()

In [ ]:
cursor_lectura.close()      # Cerramos cursores de lectura y escritura
cursor_escritura.close()

conn_lectura.close()        # Cerramos conexiones
conn_escritura.close()

In [ ]:
# Comparación de registros únicos en ambas tablas, meses y delitos
cursor.execute('SELECT count(DISTINCT id_delito) from meses')
total= cursor.fetchone()[0]
print ('Total: ', total)

Total:  2562994


In [34]:
cursor.execute('SELECT DISTINCT count(id) from delitos')
total= cursor.fetchone()[0]
print ('Total: ', total)

Total:  2562994


In [ ]:
# Creación de una tabla del orden de los meses
cursor.execute('CREATE table num_meses (Mes varchar(12), Num int not null)')
conn.commit()

In [ ]:

# Inserción de una lista de meses con número para el orden en las consultas 
meses= [('Enero', 1), ('Febrero', 2), ('Marzo', 3), ('Abril', 4), ('Mayo', 5), ('Junio', 6), ('Julio', 7), # Lista de meses
         ('Agosto', 8), ('Septiembre', 9), ('Octubre', 10), ('Noviembre', 11), ('Diciembre', 12)]
cursor.executemany('INSERT INTO num_meses (Mes, Num) values (?, ?)', meses)                              
conn.commit()

In [38]:
# Comprobación de registros
cursor.execute('SELECT * from num_meses')
rows= cursor.fetchall()
for row in rows:
    print(row)

('Enero', 1)
('Febrero', 2)
('Marzo', 3)
('Abril', 4)
('Mayo', 5)
('Junio', 6)
('Julio', 7)
('Agosto', 8)
('Septiembre', 9)
('Octubre', 10)
('Noviembre', 11)
('Diciembre', 12)


#### Consultas a la base de datos

In [6]:
# (a)Número promedio de delitos agrupados por municipio y año - mes de ocurrencia.
cursor.execute('SELECT  TOP 15 d.Municipio, d.Anio, m.Mes, avg(m.Conteo) as promedio_delitos from delitos d ' \
                'INNER JOIN meses m ON m.id_delito = d.Id '\
                'INNER JOIN num_meses n ON n.Mes = m.Mes ' \
                'GROUP BY d.Municipio, d.Anio, m.Mes, n.Mes ' \
                'ORDER BY d.Anio DESC, n.Mes ASC, promedio_delitos DESC')
rows= cursor.fetchall()
for row in rows:
    print(row)

('León', 2025, 'Abril', 44)
('Tijuana', 2025, 'Abril', 30)
('Ecatepec de Morelos', 2025, 'Abril', 29)
('Puebla', 2025, 'Abril', 28)
('Querétaro', 2025, 'Abril', 27)
('Mexicali', 2025, 'Abril', 26)
('Toluca', 2025, 'Abril', 25)
('San Luis Potosí', 2025, 'Abril', 24)
('Guadalajara', 2025, 'Abril', 24)
('Aguascalientes', 2025, 'Abril', 23)
('Iztapalapa', 2025, 'Abril', 23)
('Naucalpan de Juárez', 2025, 'Abril', 19)
('Centro', 2025, 'Abril', 19)
('Nezahualcóyotl', 2025, 'Abril', 18)
('Chihuahua', 2025, 'Abril', 18)


In [7]:
# (b)El delito más común por municipio
cursor.execute('WITH ranking AS (SELECT d.Municipio, d.Tipo_de_delito, sum(m.Conteo) as suma_conteo , ' \
                'ROW_NUMBER() OVER (PARTITION BY d.Municipio ORDER BY sum(m.Conteo) DESC) AS delito_comun ' \
                'from delitos d INNER JOIN meses m ON m.id_delito = d.Id ' \
                'GROUP BY d.Municipio, d.Tipo_de_delito) ' \
                'SELECT TOP 20 Municipio, Tipo_de_delito, suma_conteo from ranking WHERE delito_comun= 1 ' \
                'ORDER BY suma_conteo DESC')
rows= cursor.fetchall()
for row in rows:
    print(row)


('Ecatepec de Morelos', 'Robo', 230961)
('Guadalajara', 'Robo', 217031)
('Tijuana', 'Robo', 179005)
('Benito Juárez', 'Robo', 176414)
('Querétaro', 'Robo', 165817)
('Cuauhtémoc', 'Robo', 157669)
('Puebla', 'Robo', 140785)
('Mexicali', 'Robo', 138382)
('Iztapalapa', 'Robo', 137080)
('Toluca', 'Robo', 131557)
('León', 'Narcomenudeo', 121245)
('Naucalpan de Juárez', 'Robo', 120357)
('Tlalnepantla de Baz', 'Robo', 107832)
('Zapopan', 'Robo', 107375)
('Aguascalientes', 'Robo', 95367)
('Juárez', 'Violencia familiar', 94013)
('Gustavo A. Madero', 'Robo', 91682)
('Nezahualcóyotl', 'Robo', 87896)
('San Luis Potosí', 'Robo', 86952)
('Centro', 'Robo', 84412)


In [ ]:
# (c)Temporada con más crímenes en México
cursor.execute('SELECT TOP 1 d.Anio, sum(m.Conteo) as suma_conteo from delitos d ' \
                'INNER JOIN meses m ON m.id_delito = d.Id ' \
                'GROUP BY d.Anio ORDER BY suma_conteo DESC')
total= cursor.fetchone()
print('Año: ', total[0], 'Conteo: ', total[1])


Año:  2023 Conteo:  2173522


In [8]:
# (d)Número promedio de delitos en la Cd. de México. Agrupados por tipo de delito, año y mes, con orden descendente.
cursor.execute("SELECT TOP 20 d.Tipo_de_delito, d.Anio, m.Mes, avg(m.Conteo) as promedio_conteo from delitos d " \
               "INNER JOIN meses m ON m.id_delito = d.Id " \
               "INNER JOIN num_meses n ON n.Mes = m.Mes " \
               "WHERE d.Entidad = 'Ciudad de México' " \
               "GROUP BY d.Tipo_de_delito, d.Anio, m.Mes " \
               "ORDER BY promedio_conteo DESC")
rows= cursor.fetchall()
for row in rows:
    print(row)

('Violencia familiar', 2024, 'Mayo', 215)
('Violencia familiar', 2023, 'Marzo', 215)
('Violencia familiar', 2022, 'Mayo', 210)
('Violencia familiar', 2023, 'Junio', 205)
('Violencia familiar', 2024, 'Marzo', 201)
('Violencia familiar', 2023, 'Mayo', 199)
('Violencia familiar', 2022, 'Agosto', 198)
('Violencia familiar', 2024, 'Abril', 195)
('Violencia familiar', 2022, 'Marzo', 194)
('Violencia familiar', 2021, 'Marzo', 193)
('Violencia familiar', 2022, 'Octubre', 192)
('Violencia familiar', 2023, 'Abril', 191)
('Violencia familiar', 2023, 'Agosto', 189)
('Violencia familiar', 2024, 'Junio', 189)
('Violencia familiar', 2022, 'Noviembre', 189)
('Violencia familiar', 2021, 'Mayo', 188)
('Violencia familiar', 2023, 'Julio', 187)
('Violencia familiar', 2022, 'Junio', 186)
('Violencia familiar', 2025, 'Mayo', 186)
('Violencia familiar', 2023, 'Septiembre', 186)


In [ ]:
# (e)Las modalidades de delitos más frecuentes en México en los últimos 5 años
cursor.execute('SELECT top 5 d.Tipo_de_delito, d.Subtipo_de_delito, d.Modalidad, sum(m.conteo) as suma_conteo from delitos d ' \
                'INNER JOIN meses m ON m.id_delito = d.Id '\
                'WHERE d.Anio between 2020 and 2025 ' \
                'GROUP BY d.Tipo_de_delito, d.Subtipo_de_delito, d.Modalidad ' \
                'ORDER BY suma_conteo DESC')
rows= cursor.fetchall()
for row in rows:
    print(row)

('Violencia familiar', 'Violencia familiar', 'Violencia familiar', 1573546)
('Otros delitos del Fuero Común', 'Otros delitos del Fuero Común', 'Otros delitos del Fuero Común', 1183179)
('Robo', 'Otros robos', 'Sin violencia', 854176)
('Daño a la propiedad', 'Daño a la propiedad', 'Daño a la propiedad', 831914)
('Amenazas', 'Amenazas', 'Amenazas', 777958)


##### Generación de dataset: Crímenes relacionados con automóviles

In [ ]:
# Panorama general de los delitos
cursor.execute('SELECT DISTINCT Subtipo_de_delito from delitos')
rows= cursor.fetchall()
for row in rows:
    print(row)

('Robo a transeúnte en espacio abierto al público',)
('Robo en transporte individual',)
('Otros robos',)
('Abuso de confianza',)
('Violencia de género en todas sus modalidades distinta a la violencia familiar',)
('Falsedad',)
('Contra el medio ambiente',)
('Otros delitos del Fuero Común',)
('Narcomenudeo',)
('Lesiones culposas',)
('Robo en transporte público individual',)
('Corrupción de menores',)
('Lesiones dolosas',)
('Violación simple',)
('Incesto',)
('Robo de ganado',)
('Hostigamiento sexual',)
('Evasión de presos',)
('Aborto',)
('Robo a negocio',)
('Despojo',)
('Homicidio doloso',)
('Robo a casa habitación',)
('Robo de maquinaria',)
('Otros delitos contra el patrimonio',)
('Violencia familiar',)
('Incumplimiento de obligaciones de asistencia familiar',)
('Tráfico de menores',)
('Abuso sexual',)
('Robo de autopartes',)
('Amenazas',)
('Rapto',)
('Robo de vehículo automotor',)
('Robo a transportista',)
('Daño a la propiedad',)
('Otros delitos contra la familia',)
('Homicidio culposo

In [5]:
cursor.execute('SELECT DISTINCT Modalidad from delitos')
rows= cursor.fetchall()
for row in rows:
    print(row)

('Robo de embarcaciones pequeñas y grandes Sin violencia',)
('Fraude',)
('Secuestro extorsivo',)
('Con arma blanca',)
('No especificado',)
('Otros delitos que atentan contra la vida y la integridad corporal',)
('Secuestro para causar daño',)
('Narcomenudeo',)
('Con violencia',)
('Robo de embarcaciones pequeñas y grandes Con violencia',)
('Extorsión',)
('Otros delitos contra la sociedad',)
('Allanamiento de morada',)
('Robo de cables tubos y otros objetos destinados a servicios públicos Sin violencia',)
('Robo de coche de 4 ruedas Sin violencia',)
('Robo de tractores Con violencia',)
('Otros delitos contra el patrimonio',)
('Violencia familiar',)
('Incumplimiento de obligaciones de asistencia familiar',)
('Tráfico de menores',)
('Abuso sexual',)
('Robo de coche de 4 ruedas Con violencia',)
('Amenazas',)
('Rapto',)
('Sin violencia',)
('Daño a la propiedad',)
('Otros delitos contra la familia',)
('Falsedad',)
('Robo de motocicleta Sin violencia',)
('Contra el medio ambiente',)
('Otros del

In [ ]:
# Búsqueda de palabras relacionadas con automóviles
cursor.execute("SELECT DISTINCT Subtipo_de_delito, Modalidad from delitos WHERE " \
                "CONCAT(Tipo_de_delito, ' ', Subtipo_de_delito, ' ', Modalidad) LIKE '%transport%' " \
                "OR CONCAT(Tipo_de_delito, ' ', Subtipo_de_delito, ' ', Modalidad) LIKE '%tránsito%' " \
                "OR CONCAT(Tipo_de_delito, ' ', Subtipo_de_delito, ' ', Modalidad) LIKE '%auto%' " \
                "OR CONCAT(Tipo_de_delito, ' ', Subtipo_de_delito, ' ', Modalidad) LIKE '%coche%'")
rows= cursor.fetchall()
for row in rows:
    print(row)

('Robo de vehículo automotor', 'Robo de coche de 4 ruedas Sin violencia')
('Robo de vehículo automotor', 'Robo de embarcaciones pequeñas y grandes Con violencia')
('Robo en transporte individual', 'Con violencia')
('Homicidio culposo', 'En accidente de tránsito')
('Robo de autopartes', 'Sin violencia')
('Robo en transporte público individual', 'Sin violencia')
('Robo de vehículo automotor', 'Robo de motocicleta Con violencia')
('Robo en transporte público colectivo', 'Con violencia')
('Robo en transporte público individual', 'Con violencia')
('Robo de vehículo automotor', 'Robo de coche de 4 ruedas Con violencia')
('Robo de vehículo automotor', 'Robo de embarcaciones pequeñas y grandes Sin violencia')
('Robo a transportista', 'Con violencia')
('Robo en transporte individual', 'Sin violencia')
('Robo de vehículo automotor', 'Robo de motocicleta Sin violencia')
('Lesiones culposas', 'En accidente de tránsito')
('Robo a transportista', 'Sin violencia')
('Robo de autopartes', 'Con violenci

In [8]:
import csv # Importación de librería csv

In [9]:
# Generación de archivo csv
cursor.execute("Select d.Anio, d.Entidad, d.Municipio, d.Tipo_de_delito, d.Subtipo_de_delito, d.Modalidad, " \
                "m.Mes, m.Conteo from delitos d " \
                "INNER JOIN meses m on d.id= m.id_delito " \
                "INNER JOIN num_meses n ON n.Mes = m.Mes " \
                "WHERE CONCAT (d.Subtipo_de_delito, ' ', d.Modalidad) LIKE '%transport%' " \
                "OR CONCAT (d.Subtipo_de_delito, ' ', d.Modalidad) LIKE '%tránsito%' " \
                "OR CONCAT (d.Subtipo_de_delito, ' ', d.Modalidad) LIKE '%auto%' " \
                "AND Modalidad NOT LIKE '%embarcacion%' " \
                "ORDER BY d.Anio DESC, D.Entidad ASC")
rows= cursor.fetchall()                                                     # Obtiene todos los resultados
columns= [desc[0] for desc in cursor.description]                           # Obtiene nombres de columnas

with open('delitos_autos.csv', 'w', newline='', encoding= 'utf-8') as f:    # Escritura del archivo
    writer= csv.writer(f)
    writer.writerow(columns)                                                # Escribe columnas en el archivo
    writer.writerows(rows)                                                  # Escribe filas 

cursor.close()          # Cierra el cursor
conn.close()            # Cierra la conexión